# §1 Individual (unfiltered) (v12 top-50 raw-Sharpe, V4 filter)

Per-combo metrics and per-combo equity/drawdown curves on the
20% OOS test partition with no ML#2 filter. Sizing: `fixed_dollars_500` (risk $500 per trade, flat).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.evaluation._top_perf_common import (
    STARTING_EQUITY, POLICIES,
    apply_sizing, metrics_from_pnl, monte_carlo,
    load_setup,
    plot_indiv_equity, plot_indiv_dd,
    plot_combined_equity, plot_combined_dd,
    plot_ml2_indiv_equity, plot_ml2_indiv_dd,
    plot_ml2_combined_equity, plot_ml2_combined_dd,
    plot_mc_sims, plot_mc_pnl, plot_mc_sharpe, plot_mc_dd,
)

_ctx = load_setup(cost_per_contract_rt=0.0, top_strategies_path=REPO / 'evaluation' / 'top_strategies_v12_raw_sharpe_top50.json', version='v4')
bars            = _ctx['bars']
YEARS_SPAN      = _ctx['years_span']
strategies      = _ctx['strategies']
results_raw     = _ctx['results_raw']
combined_raw    = _ctx['combined_raw']
combos_ml2      = _ctx['combos_ml2']
s4_pnl_by_combo = _ctx['s4_pnl_by_combo']
ml2_portfolio   = _ctx['ml2_portfolio']


Top-K source: top_strategies_v12_raw_sharpe_top50.json


Test partition: 514,563 bars  2024-10-22 05:08:00 -> 2026-04-08 20:20:00
Years span: 1.461  (used to annualize Sharpe)
Loaded 50 strategies.
Loaded results_raw from cache (50 combos).
Combined unfiltered trades: 43,846


Loaded combos_ml2 from cache (50 combos).
ML2 portfolio trade counts: {'fixed_dollars_500': 776}


In [2]:
rows = []
for r in results_raw:
    if r['trades'].empty:
        for policy in POLICIES:
            rows.append({'combo_id': r['combo_id'], 'policy': policy,
                         **metrics_from_pnl(np.array([]), YEARS_SPAN, policy=policy)})
        continue
    t = r['trades'].sort_values('date', kind='mergesort')
    pnl_base = t['actual_pnl'].to_numpy(dtype=float)
    risk_base = t['dollar_risk'].to_numpy(dtype=float)
    r_mult = np.where(risk_base > 0, pnl_base / risk_base, 0.0)
    for policy in POLICIES:
        pnl = apply_sizing(pnl_base, risk_base, policy)
        rows.append({'combo_id': r['combo_id'], 'policy': policy,
                     **metrics_from_pnl(pnl, YEARS_SPAN, policy=policy, r=r_mult)})
perf1 = pd.DataFrame(rows)
perf1

,combo_id,policy,n_trades,trades_per_year,win_rate,total_pnl_dollars,total_return_pct,sharpe_ratio,max_drawdown_pct,max_drawdown_dollars
0,v11_7872,fixed_dollars_500,889,608.5,0.1350,45433.51,90.87,1.9295,15.50,17298.91
1,v11_23634,fixed_dollars_500,2394,1638.6,0.1316,34592.29,69.18,0.9210,29.07,30790.78
2,v11_15760,fixed_dollars_500,350,239.6,0.2057,14476.40,28.95,0.8383,13.44,10010.94
3,v11_2646,fixed_dollars_500,294,201.2,0.1667,-13728.96,-27.46,-1.2725,37.27,19084.91
4,v11_7877,fixed_dollars_500,1595,1091.7,0.3774,31254.03,62.51,1.2467,17.49,13771.60
5,v11_11149,fixed_dollars_500,286,195.8,0.4580,-5947.54,-11.90,-0.7448,16.47,8628.82
6,v11_14258,fixed_dollars_500,541,370.3,0.2366,35924.70,71.85,1.3074,15.56,14114.52
7,v11_694,fixed_dollars_500,242,165.6,0.4835,-1485.81,-2.97,-0.1986,12.69,6946.86
8,v11_9978,fixed_dollars_500,446,305.3,0.2130,23577.99,47.16,1.6803,8.63,5288.40
9,v11_8640,fixed_dollars_500,969,663.2,0.1940,16553.70,33.11,0.5295,24.66,21118.00


In [3]:
plot_indiv_equity(results_raw, 'fixed_dollars_500')

In [4]:
plot_indiv_dd(results_raw, 'fixed_dollars_500')